In [52]:
import pandas as pd
import numpy as np



In [53]:
housing = pd.read_csv('processed_data/02_data.csv')

In [54]:
housing.columns

Index(['城市', '区域', '板块', 'price', 'unit_price', '看房时间', '房屋户型', '所在楼层', '建筑面积',
       '房屋朝向', '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '挂牌时间', '交易权属', '上次交易',
       '房屋用途', '房屋年限', '产权所属', '房源标签', '核心卖点', '户型介绍', '周边配套', '交通出行', 'x',
       'y', '年份', 'nearest_park_dist_km', 'nearest_park_index',
       'nearest_park_industry', 'log_price', 'log_unit_price', 'area',
       'distance_to_center'],
      dtype='object')

In [55]:
import pandas as pd
import re

def extract_counts(text):
    # 处理空值
    if pd.isna(text):
        return pd.Series({'rooms': 0, 'halls': 0, 'kitchens': 0, 'bathrooms': 0})
    
    # 转换为字符串确保安全
    text = str(text)
    
    # 定义提取逻辑的辅助函数：找到返回数字，找不到返回0
    def get_num(pattern, string):
        match = re.search(pattern, string)
        return int(match.group(1)) if match else 0

    # 提取各项数据
    res = {
        'rooms': get_num(r'(\d+)室', text),
        'halls': get_num(r'(\d+)厅', text),
        'kitchens': get_num(r'(\d+)厨', text),
        'bathrooms': get_num(r'(\d+)卫', text)
    }
    
    return pd.Series(res)

new_cols = housing['房屋户型'].apply(extract_counts)
housing = pd.concat([housing, new_cols], axis=1)
housing = housing.drop('房屋户型', axis=1)

In [56]:
from utils import multi_label_explosion
housing = multi_label_explosion(housing, '看房时间',' ')
housing = housing.drop('看房时间', axis=1)

检测到的基础看房时间类型共有 4 种：
['只周末可看' '下班后可看' '提前预约随时可看' '有租户需预约']


In [57]:
def extract_floor(text):
    # 处理空值
    if pd.isna(text):
        return pd.Series({'floor_level': None, 'total_floors': 0})
    
    text = str(text)
    # 正则解释：
    # ^(.+?)  -> 匹配开头的文字（楼层位置）
    # \s* -> 匹配可能存在的空格
    # \(共(\d+)层\) -> 匹配 (共数字层)
    match = re.search(r'^(.+?)\s*\(共(\d+)层\)', text)
    
    if match:
        return pd.Series({
            'floor_level': match.group(1),        # 提取：中楼层
            'total_floors': int(match.group(2))   # 提取：6
        })
    else:
        # 如果格式不匹配，返回空值
        return pd.Series({'floor_level': None, 'total_floors': 0})

new_floor_cols = housing['所在楼层'].apply(extract_floor)
housing = pd.concat([housing, new_floor_cols], axis=1)
housing = housing.drop('所在楼层', axis=1)
multi_label_explosion(housing,'floor_level',' ')
housing = housing.drop('floor_level', axis=1)

检测到的基础floor_level类型共有 6 种：
['低楼层' '底层' '高楼层' '中楼层' '顶层' '地下室']


In [58]:
multi_label_explosion(housing,'房屋朝向',' ')
housing = housing.drop('房屋朝向', axis=1)


检测到的基础房屋朝向类型共有 8 种：
['南' '北' '东' '西' '西北' '西南' '东南' '东北']


In [59]:
from utils import process_num

housing['建筑面积'] = housing['建筑面积'].apply(process_num)

In [60]:
housing['建筑结构'] = housing['建筑结构'].fillna('未知结构')
multi_label_explosion(housing, '建筑结构',' ')
housing = housing.drop('建筑结构', axis=1)

检测到的基础建筑结构类型共有 7 种：
['钢混结构' '混合结构' '未知结构' '砖混结构' '钢结构' '砖木结构' '框架结构']


In [61]:
housing['装修情况'] = housing['装修情况'].fillna('其它')
multi_label_explosion(housing, '装修情况',' ')
housing = housing.drop('装修情况', axis=1)

检测到的基础装修情况类型共有 5 种：
['精装' '简装' '毛坯' '其它' '其他']


In [62]:
from utils import cn_to_int_custom

def extract_elevator_household_final(text):
    res = {'elevators': 0, 'households': 0}
    if pd.isna(text):
        return pd.Series(res)
    
    text = str(text)
    # 正则提取：匹配“梯”前面的中文和“户”前面的中文
    # 范围：一到九、两、十、百、零
    pattern = r'([一二三四五六七八九十百零两]+)梯([一二三四五六七八九十百零两]+)户'
    match = re.search(pattern, text)
    
    if match:
        res['elevators'] = cn_to_int_custom(match.group(1))
        res['households'] = cn_to_int_custom(match.group(2))
        
    return pd.Series(res)

new_cols = housing['梯户比例'].apply(extract_elevator_household_final)
housing = pd.concat([housing, new_cols], axis=1)
housing = housing.drop('梯户比例', axis=1)

housing['elevators_household_ratio'] = housing.apply(lambda row: row['elevators'] / row['households'] if row['households'] > 0 else 0, axis=1)

In [63]:
housing['配备电梯'] = housing['配备电梯'].map({'有': 1, '无': 0})
housing['配备电梯'] = housing['配备电梯'].fillna(housing['elevators']>0).astype(int)

In [64]:
housing['别墅类型'] = housing['别墅类型'].fillna('非别墅')
multi_label_explosion(housing, '别墅类型', ' ')
housing = housing.drop('别墅类型', axis=1)

检测到的基础别墅类型类型共有 5 种：
['非别墅' '独栋' '联排' '叠拼' '双拼']


In [65]:
multi_label_explosion(housing, '交易权属',' ')
housing = housing.drop('交易权属', axis=1)

检测到的基础交易权属类型共有 9 种：
['私产' '商品房' '已购公房' '一类经济适用房' '二类经济适用房' '央产房' '限价商品房' '定向安置房' '自住型商品房']


In [66]:
multi_label_explosion(housing, '房屋用途','/')
housing = housing.drop('房屋用途', axis=1)

检测到的基础房屋用途类型共有 14 种：
['普通住宅' '别墅' '车库' '商业办公类' '公寓' '住宅' '酒店式公寓' '平房' '商务型公寓' '公寓（住宅）' '服务式公寓'
 '四合院' '住宅式公寓' '单身公寓（住宅）']


In [67]:
multi_label_explosion(housing, '房屋年限',' ')
housing = housing.drop('房屋年限', axis=1)

检测到的基础房屋年限类型共有 3 种：
['满五年' '满两年' '未满两年']


In [68]:
housing['产权所属'] = housing['产权所属'].map({'非共有':2, '共有':1}).fillna(0).astype(int)


In [69]:
multi_label_explosion(housing, '房源标签','、')
housing = housing.drop('房源标签', axis=1)

检测到的基础房源标签类型共有 6 种：
['VR看装修' '房本满五年' 'VR房源' '地铁' '随时看房' '房本满两年']


In [70]:
housing.columns

Index(['城市', '区域', '板块', 'price', 'unit_price', '建筑面积', '配备电梯', '挂牌时间', '上次交易',
       '产权所属', '核心卖点', '户型介绍', '周边配套', '交通出行', 'x', 'y', '年份',
       'nearest_park_dist_km', 'nearest_park_index', 'nearest_park_industry',
       'log_price', 'log_unit_price', 'area', 'distance_to_center', 'rooms',
       'halls', 'kitchens', 'bathrooms', '看房时间_is_只周末可看', '看房时间_is_下班后可看',
       '看房时间_is_提前预约随时可看', '看房时间_is_有租户需预约', 'total_floors',
       'floor_level_is_低楼层', 'floor_level_is_底层', 'floor_level_is_高楼层',
       'floor_level_is_中楼层', 'floor_level_is_顶层', 'floor_level_is_地下室',
       '房屋朝向_is_南', '房屋朝向_is_北', '房屋朝向_is_东', '房屋朝向_is_西', '房屋朝向_is_西北',
       '房屋朝向_is_西南', '房屋朝向_is_东南', '房屋朝向_is_东北', '建筑结构_is_钢混结构',
       '建筑结构_is_混合结构', '建筑结构_is_未知结构', '建筑结构_is_砖混结构', '建筑结构_is_钢结构',
       '建筑结构_is_砖木结构', '建筑结构_is_框架结构', '装修情况_is_精装', '装修情况_is_简装',
       '装修情况_is_毛坯', '装修情况_is_其它', '装修情况_is_其他', 'elevators', 'households',
       'elevators_household_ratio', '别墅类型_is_非别墅', '别墅类型_is_独栋', '别墅类型_

In [71]:
housing.to_csv('processed_data/03_data.csv', index=False)